In [ ]:
import faiss
import numpy as np
import time

from utils import utils

In [ ]:
# load vector
TENSOR_DIR = "embeddings/chunked_6seconds/"
tensors = utils.load_tensor(TENSOR_DIR, num_workers=16, use_cache=False)


In [ ]:
# estimate total # of tensors
total_tensors = 0
for k, v in tensors.items():
  total_tensors += v.shape[0]

print(f"Total tensors: {total_tensors}")

In [ ]:
# FAISS Index Config
d = 1024
nb = total_tensors
nt = 638_976
batch_size = 100_000
index_string = "OPQ64_1024,IVF16384,PQ128"


In [ ]:

# --- 1. Setup GPU Resources ---
# (Optional) If you have multiple GPUs, you can use index_cpu_to_all_gpus
# res = faiss.StandardGpuResources()

# --- 2. Initialize the Index (CPU first) ---
# We build the structure on CPU, then move to GPU for training/indexing
index_cpu = faiss.index_factory(d, index_string)
index_cpu.verbose = True

# Move to GPU
# Since the COMPRESSED index will be ~640MB, it fits easily in VRAM.
# print("Moving index to GPU...")
# index = faiss.index_cpu_to_gpu(res, 0, index_cpu)

index = index_cpu

In [ ]:
# --- 3. Training ---
# You cannot add data to an IVF/PQ index without training it first.
# We generate random data for training (replace this with your actual data sample)
print(f"Generating {nt} training vectors...")
xt = np.random.random((nt, d)).astype('float32')

print("Training index (this may take a few minutes)...")
start_time = time.time()
index.train(xt)
print(f"Training done in {time.time() - start_time:.2f}s")

# Free memory from training data
del xt


In [ ]:
# --- 4. Adding Data (in Batches) ---
print(f"Adding {nb} vectors in batches of {batch_size}...")
for i in range(0, nb, batch_size):
    # Simulate data loading (Replace with your actual data loader)
    # Ensure data is float32
    xb_batch = np.random.random((batch_size, d)).astype('float32')
    
    index.add(xb_batch)
    if i % 1_000_000 == 0:
        print(f"  Indexed {i} / {nb}...")

print(f"Total vectors indexed: {index.ntotal}")


In [ ]:
# --- 5. Searching ---
k = 5      # Nearest neighbors to return
nprobe = 32 # How many clusters to visit (Speed vs Accuracy trade-off)

# Set nprobe (runtime parameter)
# Note: For GpuIndex, we use GpuParameterSpace or set it before query
# But standard faiss.IndexIVF methods often work if wrapped correctly.
# The surest way on GPU is usually constructing the search params or accessing the object options.
# However, standard property setting often propagates:
index.nprobe = nprobe 

print("Running test search...")
xq = np.random.random((1, d)).astype('float32') # Query vector
D, I = index.search(xq, k)

print("Distances:", D)
print("Indices:", I)

# --- 6. Saving (Move back to CPU to save) ---
print("Saving index to disk...")
index_cpu_final = faiss.index_gpu_to_cpu(index)
faiss.write_index(index_cpu_final, "large_index_10m.faiss")
print("Saved to large_index_10m.faiss")